# Task 04 — Segmentation and Recommendation Model
**Objective:** Segment customers by behavior and create an explainable, reproducible churn-risk score.

Dataset: 15 synthetic customer records. Monthly charges are treated as fictional INR amounts, as stated in the EdVyro analytics source guide.

The score is a prioritisation heuristic, not a trained model or probability. The observed `Churn` label is excluded from score calculation and used only for descriptive comparison after scoring.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv("customer_churn_sample.csv")
print("Shape:", df.shape)
df.head()


## 1. Data quality checks
Check missing values, duplicates, duplicate IDs and impossible negative scoring values before scoring.


In [ ]:
required=["CustomerID","ContractType","TenureMonths","SupportTickets","MonthlyCharges","Churn"]
assert set(required).issubset(df.columns)
print("Missing values:\\n",df[required].isna().sum())
print("Duplicate rows:",df.duplicated().sum())
print("Duplicate IDs:",df.CustomerID.duplicated().sum())
invalid=(df.TenureMonths<0)|(df.SupportTickets<0)|(df.MonthlyCharges<0)
print("Invalid negative-value rows:",int(invalid.sum()))


## 2. Feature selection and scoring rules

| Feature | Rule | Points |
|---|---|---:|
| Contract type | Month-to-Month | +3 |
| Tenure | 12 months or less | +2 |
| Support tickets | 4 or more | +2 |
| Monthly charges | INR 80 or more | +1 |

Maximum score = 8. Low = 0–2; Medium = 3–4; High = 5–8. Weights are transparent business assumptions, not statistically estimated coefficients. Gender and age are not used. Churn is excluded to avoid target leakage.


In [ ]:
df["contract_points"]=df.ContractType.eq("Month-to-Month").astype(int)*3
df["tenure_points"]=df.TenureMonths.le(12).astype(int)*2
df["support_points"]=df.SupportTickets.ge(4).astype(int)*2
df["account_value_points"]=df.MonthlyCharges.ge(80).astype(int)
parts=["contract_points","tenure_points","support_points","account_value_points"]
df["RiskScore"]=df[parts].sum(axis=1)
df["RiskTier"]=pd.cut(df.RiskScore,[-1,2,4,8],labels=["Low","Medium","High"]).astype(str)
df["CustomerSegment"]=df.RiskTier.map({"Low":"Stable contracted customers","Medium":"Contract watchlist","High":"High-priority customers"})
df[["CustomerID","RiskScore","RiskTier","CustomerSegment"]]


## 3. Explain individual scores
The reason field identifies exactly which rules contributed points. A score of zero means no listed rule fired; it does not mean zero true churn probability.


In [ ]:
def explain(row):
    reasons=[]
    if row.contract_points: reasons.append("Month-to-Month contract (+3)")
    if row.tenure_points: reasons.append("Tenure <= 12 months (+2)")
    if row.support_points: reasons.append("Support tickets >= 4 (+2)")
    if row.account_value_points: reasons.append("Monthly charges >= INR 80 (+1)")
    return "; ".join(reasons) if reasons else "No scored risk signals"
df["RiskReasons"]=df.apply(explain,axis=1)


## 4. Segment summary and descriptive validation
Compare risk tiers with observed churn only after score calculation. This is an in-sample description of 15 records, not out-of-sample predictive validation.


In [ ]:
segment_summary=df.groupby(["CustomerSegment","RiskTier"],observed=True).agg(
    CustomerCount=("CustomerID","count"),
    AverageRiskScore=("RiskScore","mean"),
    MonthlyChargesTotal=("MonthlyCharges","sum"),
    ObservedChurners=("Churn",lambda s:(s=="Yes").sum())
).reset_index()
segment_summary["ObservedChurnRatePct"]=(100*segment_summary.ObservedChurners/segment_summary.CustomerCount).round(1)
segment_summary


In [ ]:
risk_validation=df.groupby("RiskTier",observed=True).agg(
    Customers=("CustomerID","count"),
    ObservedChurners=("Churn",lambda s:(s=="Yes").sum()),
    RetainedCustomers=("Churn",lambda s:(s=="No").sum())
)
risk_validation["ObservedChurnRatePct"]=(100*risk_validation.ObservedChurners/risk_validation.Customers).round(1)
risk_validation


## 5. Visualise the risk tiers

In [ ]:
counts=df.RiskTier.value_counts().reindex(["Low","Medium","High"],fill_value=0)
ax=counts.plot(kind="bar",title="Customer count by churn-risk tier",xlabel="Risk tier",ylabel="Customers")
ax.tick_params(axis="x",rotation=0)
plt.tight_layout()
plt.show()


## 6. Edge-case validation
Expected boundaries: minimum 0, maximum 8; tenure exactly 12 triggers +2; tickets exactly 4 triggers +2; charge exactly INR 80 triggers +1. Missing or negative scoring inputs must be rejected for review, not silently converted to zero.


In [ ]:
def score_customer(contract,tenure,tickets,monthly_charge):
    vals=[contract,tenure,tickets,monthly_charge]
    if any(pd.isna(v) for v in vals):
        raise ValueError("Missing scoring input: data-quality review required.")
    if tenure<0 or tickets<0 or monthly_charge<0:
        raise ValueError("Negative scoring input is invalid.")
    return (3 if contract=="Month-to-Month" else 0)+(2 if tenure<=12 else 0)+(2 if tickets>=4 else 0)+(1 if monthly_charge>=80 else 0)

assert score_customer("Two Year",13,0,49.99)==0
assert score_customer("Month-to-Month",12,4,80)==8
assert score_customer("Month-to-Month",12,0,49.99)==5
assert score_customer("One Year",13,4,49.99)==2
assert score_customer("One Year",13,0,80)==1
for bad in [("One Year",np.nan,0,50),("One Year",-1,0,50)]:
    try: score_customer(*bad)
    except ValueError: print("Correctly rejected:",bad)
print("Boundary tests passed.")


## 7. Retention recommendations and limitations
- **High-priority customers:** review support history, resolve open issues and offer a relevant retention conversation. Measure contact rate, issue resolution and subsequent retention.
- **Contract watchlist:** check service concerns and explain longer-contract options without pressure. Measure response and renewal conversion.
- **Stable contracted customers:** maintain service quality, send renewal reminders and monitor new support-ticket spikes. Measure renewal and complaint rates.

Only 15 synthetic rows are available. The score is heuristic; product usage, payment failures, satisfaction and time-series engagement are absent. In-sample alignment does not prove predictive accuracy or causation. Validate on a larger later-period dataset and compare contacted customers with a holdout group.
